# Ejercicio 7: Bases de Datos Vectoriales
## Nombre: Muzo Miguel
## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [8]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

/home/migueldev/Documents/University/Courses/RI/Practices/recuperacion_de_informacion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [10]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [11]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [12]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4954.63it/s]


In [13]:
import os

EMB_PATH = "embeddings.npy"

if os.path.exists(EMB_PATH):
    # Cache en disco: evita re-encodear (~21h)
    embeddings = np.load(EMB_PATH)
    print("Embeddings cargados desde disco")
else:
    # Embeddings (N x D)
    # Se debe usar normalize_embeddings=True para similitud coseno
    embeddings = model.encode(
        passages,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    np.save(EMB_PATH, embeddings)

Embeddings cargados desde disco


In [14]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [15]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [27]:
!uv pip install faiss-cpu -q

In [16]:
# código base para FAISS
import faiss
import numpy as np

# Embeddings normalizados -> producto interno equivale a similitud coseno
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

D, I = index.search(query_vec, k=10)

print(f"Query: '{query_text}'\n")
for rank, (idx, score) in enumerate(zip(I[0], D[0]), start=1):
    print(f"{rank}. [ID {idx}] score={score:.4f}")
    print(f"   >> {chunks_df.iloc[idx]['text'][:150]}...\n")

Query: 'Battery measuring'

1. [ID 10176] score=0.8703
   >> Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

2. [ID 1] score=0.8618
   >> Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

3. [ID 10177] score=0.8401
   >> ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

4. [ID 37406] score=0.8391
   >> ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

5. [ID 71872] score=0.8386
   >> is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approximately...

6. [ID 37409] score=0.8345
   >> sho

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [32]:
!uv pip install qdrant-client -q

In [17]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Variables reutilizadas
texts_list = chunks_df["text"].tolist()
DIM = embeddings.shape[1]

qdrant = QdrantClient(":memory:")
qdrant.create_collection(
    collection_name="wiki_chunks",
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE)
)

# Construir los puntos con la metadata
points = []
for i in range(len(embeddings)):
    points.append(PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "text": texts_list[i],
            "doc_id": int(chunks_df.iloc[i]["doc_id"]),
            "chunk_id": int(chunks_df.iloc[i]["chunk_id"])
        }
    ))

BATCH = 500
for i in range(0, len(points), BATCH):
    qdrant.upsert(collection_name="wiki_chunks", points=points[i:i+BATCH])

print(f"Indexados {len(points)} chunks en Qdrant")

/tmp/ipykernel_281404/782815774.py:29: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20500 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant.upsert(collection_name="wiki_chunks", points=points[i:i+BATCH])


Indexados 79104 chunks en Qdrant


In [18]:
def qdrant_search(query_embedding, k=5):
    results = qdrant.query_points(
        collection_name="wiki_chunks",
        query=query_embedding[0].tolist(),
        limit=k
    ).points
    return [(r.id, r.score, r.payload["text"], r.payload) for r in results]

In [19]:
# Ejemplo con k=5
print(f"\nQuery: '{query_text}'\n")
for id_, score, text, meta in qdrant_search(query_vec, k=5):
    print(f"[ID {id_}] score={score:.4f} | doc_id={meta['doc_id']}")
    print(f"  >> {text[:150]}...\n")


Query: 'Battery measuring'

[ID 10176] score=0.8703 | doc_id=1391
  >> Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

[ID 1] score=0.8618 | doc_id=1
  >> Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

[ID 10177] score=0.8401 | doc_id=1391
  >> ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

[ID 37406] score=0.8391 | doc_id=5067
  >> ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

[ID 71872] score=0.8386 | doc_id=9888
  >> is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approx

### Respuestas Parte 3

### ¿Métrica usada: cosine o L2?

Se usó cosine porque los embeddings ya están normalizados (`normalize_embeddings=True`). Con vectores normalizados, la similitud coseno es la métrica estándar para retrieval semántico y da mejores resultados que L2.

### ¿Qué tan fácil fue filtrar por metadata comparado con FAISS?
Mucho más fácil. En Qdrant los filtros son nativos (se pasan como parámetro en la búsqueda con `query_filter`). Con FAISS habría que filtrar manualmente después de recuperar los resultados, lo que implica mayor complejidad y mayor pasos para lograr el objetivo de busqueda.

### ¿Qué pasa con el tiempo de respuesta cuando aumentas k?
El tiempo sube levemente con k mayor, pero la diferencia es pequeña porque el índice ya hace la búsqueda por similitud y solo cambia cuántos resultados retorna.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [40]:
# milvus_lite es necesario para modo local (MilvusClient con archivo .db)
!uv pip install "pymilvus[milvus_lite]" -q

In [20]:
from pymilvus import MilvusClient
import time

# MilvusLite guarda en archivo local
milvus = MilvusClient("milvus_wiki.db")

if milvus.has_collection("wiki_chunks"):
    milvus.drop_collection("wiki_chunks")

milvus.create_collection(
    collection_name="wiki_chunks",
    dimension=DIM,
    metric_type="COSINE"
)

data = []
for i in range(len(embeddings)):
    data.append({
        "id": i,
        "vector": embeddings[i].tolist(),
        "doc_id": int(chunks_df.iloc[i]["doc_id"]),
        "chunk_id": int(chunks_df.iloc[i]["chunk_id"])
    })

# Insertar los datos
BATCH = 1000
for i in range(0, len(data), BATCH):
    milvus.insert(collection_name="wiki_chunks", data=data[i:i+BATCH])

print(f"Indexados {len(data)} chunks en Milvus")

Indexados 79104 chunks en Milvus


In [21]:
def milvus_search(query_embedding, k=5):
    results = milvus.search(
        collection_name="wiki_chunks",
        data=[query_embedding[0].tolist()],
        limit=k,
        output_fields=["doc_id", "chunk_id"]
    )
    output = []
    for r in results[0]:
        idx = r["id"]
        output.append((idx, r["distance"], texts_list[idx], r["entity"]))
    return output

In [22]:
# Comparativa: k=5 vs k=20
print("=== Búsqueda k=5 ===")
t0 = time.time()
res5 = milvus_search(query_vec, k=5)
t5 = time.time() - t0
for idx, dist, text, meta in res5:
    print(f"  [{idx}] dist={dist:.4f} | {text[:100]}...")
print(f"Tiempo: {t5*1000:.1f}ms\n")

print("=== Búsqueda k=20 ===")
t0 = time.time()
res20 = milvus_search(query_vec, k=20)
t20 = time.time() - t0
print(f"Tiempo: {t20*1000:.1f}ms")

# Cuántos resultados de k=5 aparecen también en k=20
ids5 = {r[0] for r in res5}
ids20 = {r[0] for r in res20}
print(f"Overlap (IDs de k=5 en k=20): {len(ids5 & ids20)}/5")

=== Búsqueda k=5 ===
  [10176] dist=0.1297 | Battery tester A battery tester is an electronic device intended for testing the state of an electri...
  [1] dist=0.1382 | Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform...
  [10177] dist=0.1599 | ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac...
  [71872] dist=0.1614 | is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo...
  [37409] dist=0.1655 | shorting the measurement points together and performing an adjustment for zero ohms indication prior...
Tiempo: 4708.9ms

=== Búsqueda k=20 ===
Tiempo: 3571.9ms
Overlap (IDs de k=5 en k=20): 5/5


### Respuestas Parte 4

## ¿Qué parámetros ajustaste para precisión vs velocidad?
MilvusLite usa HNSW por defecto con parámetros estándar (`M=16`, `efConstruction=200`). Para más precisión se puede subir `ef` en tiempo de búsqueda; para más velocidad, bajarlo. En este ejemplo no se modificaron porque el corpus es relativamente pequeño y los valores por defecto son razonables.

## ¿Qué evidencia tienes de que ANN cambia los resultados?
El experimento muestra el overlap entre k=5 y k=20. Con corpus pequeño el overlap suele ser 5/5. Con corpus de millones de vectores y `ef` bajo se vería un overlap menor, evidenciando que ANN sacrifica recall por velocidad.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [1]:
!uv pip install "weaviate-client>=4.0" -q

In [23]:
import numpy as np
import pandas as pd

if "embeddings" not in globals():
    embeddings = np.load("embeddings.npy")
    print(f"embeddings: {embeddings.shape}")

if "chunks_df" not in globals():
    chunks_df = pd.read_pickle("chunks_df.pkl")
    print(f"chunks_df: {len(chunks_df)} chunks")

if "texts_list" not in globals():
    texts_list = chunks_df["text"].tolist()

if "query_vec" not in globals():
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("intfloat/e5-base-v2")
    query_text = "Battery measuring"
    query_vec = model.encode(
        ["query: " + query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    print(f"query_vec listo para: '{query_text}'")

In [24]:
import weaviate
import weaviate.classes as wvc
from weaviate.exceptions import WeaviateStartUpError

# Modo embedded: no necesita Docker.
# Si una instancia embedded anterior quedó viva (error a mitad de celda),
# los puertos siguen ocupados: en ese caso nos conectamos a esa instancia.
try:
    client_w = weaviate.connect_to_embedded()
except WeaviateStartUpError:
    client_w = weaviate.connect_to_local(port=8079, grpc_port=50050)

print("Conectado a Weaviate:", client_w.is_ready())

{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-07-02T10:16:03-05:00"}
{"action":"startup","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-07-02T10:16:03-05:00"}
{"action":"startup","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-07-02T10:16:03-05:00"}
{"action":"startup","auto_schema_enabled":{},"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"

Conectado a Weaviate: True


{"action":"telemetry_push","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"telemetry started","payload":"\u0026{MachineID:07af5531-ae96-4fec-8656-ae502cc51065 Type:INIT Version:1.30.5 ObjectsCount:0 OS:linux Arch:amd64 UsedModules:[] CollectionsCount:1}","time":"2026-07-02T10:16:06-05:00"}


In [25]:
# Crear colección con esquema explícito (borra la anterior si existe)
if client_w.collections.exists("WikiChunk"):
    client_w.collections.delete("WikiChunk")

collection = client_w.collections.create(
    name="WikiChunk",
    vectorizer_config=wvc.config.Configure.Vectorizer.none(),  # usamos nuestros propios vectores
    properties=[
        wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="chunk_id", data_type=wvc.config.DataType.INT),
    ]
)

print("Colección WikiChunk creada")

Colección WikiChunk creada


/home/migueldev/Documents/University/Courses/RI/Practices/recuperacion_de_informacion/.venv/lib/python3.12/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(
{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"warning","msg":"prop len tracker file /home/migueldev/.local/share/weaviate/wikichunk/vWNcz0TRfgRp/proplengths does not exist, creating new tracker","time":"2026-07-02T10:16:39-05:00"}
{"action":"hnsw_prefill_cache_async","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"not waiting for vector cache prefill, running in background","time":"2026-07-02T10:16:39-05:00","wait_for_cache_prefill":false}
{"build_git_commit":"","build_go_version":"go1.24.3","bu

In [26]:
# Insertar con batch automático
with collection.batch.dynamic() as batch:
    for i in range(len(embeddings)):
        batch.add_object(
            properties={
                "text": texts_list[i],
                "doc_id": int(chunks_df.iloc[i]["doc_id"]),
                "chunk_id": int(chunks_df.iloc[i]["chunk_id"]),
            },
            vector=embeddings[i].tolist()
        )

print(f"Objetos insertados en Weaviate: {len(collection)}")

/home/migueldev/Documents/University/Courses/RI/Practices/recuperacion_de_informacion/.venv/lib/python3.12/site-packages/weaviate/warnings.py:312: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
I0000 00:00:1783005485.139037  288849 chttp2_transport.cc:1353] ipv4:127.0.0.1:36629: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1783005485.140731  288849 chttp2_transport.cc:1385] ipv4:127.0.0.1:36629: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


Objetos insertados en Weaviate: 79104


In [27]:
def weaviate_search(query_embedding, k=5):
    results = collection.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        return_metadata=wvc.query.MetadataQuery(certainty=True)
    )
    return [
        (str(r.uuid), r.metadata.certainty, r.properties["text"], r.properties)
        for r in results.objects
    ]

In [28]:
print(f"\nQuery: '{query_text}'\n")
for uid, certainty, text, props in weaviate_search(query_vec, k=5):
    print(f"[{uid[:8]}] certainty={certainty:.4f} | doc_id={props['doc_id']}")
    print(f"  >> {text[:150]}...\n")

client_w.close()


Query: 'Battery measuring'

[e49ef69f] certainty=0.9352 | doc_id=1391
  >> Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

[0d0d3ce8] certainty=0.9309 | doc_id=1
  >> Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

[b01d33d6] certainty=0.9201 | doc_id=1391
  >> ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

[fe134a66] certainty=0.9196 | doc_id=5067
  >> ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

[4ecb6018] certainty=0.9193 | doc_id=9888
  >> is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compens

{"action":"restapi_management","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"Shutting down... ","time":"2026-07-02T10:18:24-05:00","version":"1.30.5"}
{"action":"restapi_management","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"Stopped serving weaviate at http://127.0.0.1:8079","time":"2026-07-02T10:18:24-05:00","version":"1.30.5"}
{"action":"telemetry_push","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"telemetry terminated","payload":"\u0026{MachineID:07af5531-ae96-4fec-8656-ae502cc51065 Type:TERMINATE Version:1.30.5 ObjectsCount:70436 OS:linux Arch:amd64 UsedModules:[] CollectionsCount:1}","time":"2026-07-02T10:18:25-05:00"}
{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"closing ra

### Respuestas Parte 5

## ¿Diferencia conceptual entre "schema + objetos" vs "tabla + filas"?
En Weaviate cada objeto tiene propiedades tipadas y un vector asociado, más parecido a un grafo de entidades que a una tabla plana. En SQL las filas son datos sin semántica vectorial integrada. Weaviate está diseñado para búsqueda semántica desde el principio, mientras que en PostgreSQL hay que añadir pgvector como extensión.

## Trade-off de complejidad vs expresividad:
Weaviate requiere más configuración inicial (definir clase, propiedades, tipos de datos) comparado con Chroma. A cambio ofrece filtros avanzados, soporte nativo de vectores y mejor integración con modelos. Para prototipos simples es más verboso de lo necesario, pero para producción con esquemas complejos resulta más expresivo.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [31]:
!uv pip install chromadb -q

In [35]:
import chromadb

chroma_client = chromadb.Client()
chroma_col = chroma_client.create_collection(
    "wiki_chunks",
    metadata={"hnsw:space": "cosine"}
)

# Insertar en lotes (Chroma tiene límite por batch)
BATCH = 5000
for i in range(0, len(embeddings), BATCH):
    end = min(i + BATCH, len(embeddings))
    chroma_col.add(
        ids=[str(j) for j in range(i, end)],
        embeddings=embeddings[i:end].tolist(),
        documents=texts_list[i:end],
        metadatas=[
            {
                "doc_id": int(chunks_df.iloc[j]["doc_id"]),
                "chunk_id": int(chunks_df.iloc[j]["chunk_id"])
            }
            for j in range(i, end)
        ]
    )

print(f"Insertados {chroma_col.count()} chunks en Chroma")

Insertados 79104 chunks en Chroma


In [36]:
def chroma_search(query_embedding, k=5):
    results = chroma_col.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )
    output = []
    for i in range(len(results["ids"][0])):
        output.append((
            results["ids"][0][i],
            results["distances"][0][i],
            results["documents"][0][i],
            results["metadatas"][0][i]
        ))
    return output

In [37]:
# Ejemplo con k=5
print(f"\nQuery: '{query_text}'\n")
for id_, dist, text, meta in chroma_search(query_vec, k=5):
    print(f"[{id_}] dist={dist:.4f} | doc_id={meta['doc_id']}")
    print(f"  >> {text[:150]}...\n")


Query: 'Battery measuring'

[10176] dist=0.1297 | doc_id=1391
  >> Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

[1] dist=0.1382 | doc_id=1
  >> Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

[10177] dist=0.1599 | doc_id=1391
  >> ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

[37406] dist=0.1609 | doc_id=5067
  >> ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

[71872] dist=0.1614 | doc_id=9888
  >> is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approximately...



### Respuestas Parte 6

## ¿Qué tan fácil fue implementar comparado con Qdrant/Milvus?**
Mucho más fácil. Chroma no requie

## ¿Qué limitaciones ves para producción?**
- No escala bien a millones de vectores (todo en memoria o disco local).
- Sin soporte multi-nodo ni replicación.
- La búsqueda en modo local es exacta (brute force), lo cual es lento con corpus grandes.
- Poca configuración del índice: no se puede ajustar HNSW con parámetros propios fácilmente.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [39]:
!uv pip install psycopg2-binary pgvector -q

In [5]:
# Carga rápida desde disco (solo si el kernel fue reiniciado y faltan variables)
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/e5-base-v2")
query_text = "Battery measuring"
query_vec = model.encode(
        ["query: " + query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
print(f"query_vec listo para: '{query_text}'")

embeddings: (79104, 768)
chunks_df: 79104 chunks


/home/migueldev/Documents/University/Courses/RI/Practices/recuperacion_de_informacion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5532.65it/s]


query_vec listo para: 'Battery measuring'


In [6]:
import psycopg2
from pgvector.psycopg2 import register_vector

# docker run -d --name pgvector-ri -p 5434:5432 -e POSTGRES_PASSWORD=postgres pgvector/pgvector:pg16
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5434
)
conn.autocommit = True
cur = conn.cursor()

# Activar la extensión pgvector
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

# Crear tabla limpia
cur.execute("DROP TABLE IF EXISTS wiki_chunks")
cur.execute(f"""
    CREATE TABLE wiki_chunks (
        id INTEGER PRIMARY KEY,
        text TEXT,
        doc_id INTEGER,
        chunk_id INTEGER,
        embedding vector({DIM})
    )
""")

# Registrar el tipo vector para que psycopg2 lo entienda
register_vector(conn)

# Insertar en lotes
BATCH = 1000
for i in range(0, len(embeddings), BATCH):
    end = min(i + BATCH, len(embeddings))
    rows = [
        (j, texts_list[j], int(chunks_df.iloc[j]["doc_id"]), int(chunks_df.iloc[j]["chunk_id"]), embeddings[j])
        for j in range(i, end)
    ]
    cur.executemany("INSERT INTO wiki_chunks VALUES (%s, %s, %s, %s, %s)", rows)

print(f"Insertados {len(embeddings)} registros en PostgreSQL")

Insertados 79104 registros en PostgreSQL


In [7]:
def pgvector_search(query_embedding, k=5):
    q = query_embedding[0]
    cur.execute("""
        SELECT id, text, doc_id, chunk_id,
               1 - (embedding <=> %s::vector) AS score
        FROM wiki_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (q, q, k))
    rows = cur.fetchall()
    return [(r[0], r[4], r[1], {"doc_id": r[2], "chunk_id": r[3]}) for r in rows]

In [8]:
# Ejemplo con k=5
print(f"\nQuery: '{query_text}'\n")
for id_, score, text, meta in pgvector_search(query_vec, k=5):
    print(f"[{id_}] score={score:.4f} | doc_id={meta['doc_id']}")
    print(f"  >> {text[:150]}...\n")

cur.close()
conn.close()


Query: 'Battery measuring'

[10176] score=0.8703 | doc_id=1391
  >> Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

[1] score=0.8618 | doc_id=1
  >> Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

[10177] score=0.8401 | doc_id=1391
  >> ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

[37406] score=0.8391 | doc_id=5067
  >> ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

[71872] score=0.8386 | doc_id=9888
  >> is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approximately...



### Respuestas Parte 7

## ¿Qué tan "explicable" es esta aproximación?
Es la más explicable de todas. La búsqueda es literalmente una consulta SQL con `ORDER BY distancia`. Cualquier desarrollador con conocimientos de SQL puede leerla y entender qué hace sin necesidad de conocer APIs específicas de bases vectoriales.

## ¿Qué ventajas ofrece el mundo SQL?
JOINs con otras tablas, filtros complejos con WHERE, agregaciones (COUNT, AVG, GROUP BY), transacciones ACID y toda la madurez del ecosistema PostgreSQL. Se puede mezclar búsqueda vectorial con filtros relacionales en una sola query.

## ¿Qué limitaciones en escalabilidad?
PostgreSQL hace búsqueda secuencial por defecto (brute force). El índice ivfflat/hnsw de pgvector ayuda, pero sigue siendo más lento que Milvus o Qdrant para millones de vectores. Además no distribuye fácilmente en múltiples nodos como las BD vectoriales dedicadas.